<a href="https://colab.research.google.com/github/quevedooo/PROGCOM_B_2026/blob/main/Nuevo_proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install streamlit pandas openpyxl fpdf pyngrok

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.2 MB/s eta 0:00:00
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=298ed9ec75e926b334dfe45eb746c86c89b73c44c8e0d4c14395d4ed7aea3a1a
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [12]:
%%writefile app.py

import streamlit as st
import pandas as pd
from fpdf import FPDF
import tempfile

st.set_page_config(
    page_title="VAN Pro Analyzer",
    layout="wide"
)

# =========================
# ESTILOS
# =========================
st.markdown("""
<style>

.main {
    background-color: #0E1117;
}

.big-title {
    font-size: 42px;
    font-weight: bold;
    color: #4CAF50;
}

.result-box {
    padding: 20px;
    border-radius: 15px;
    background-color: #1E1E1E;
    color: white;
    font-size: 24px;
    text-align: center;
    margin-top:20px;
}

</style>
""", unsafe_allow_html=True)

# =========================
# TITULO
# =========================
st.markdown(
    '<p class="big-title">📊 VAN Pro Analyzer</p>',
    unsafe_allow_html=True
)

st.write("Calculadora financiera profesional")

# =========================
# SIDEBAR
# =========================
st.sidebar.title("⚙️ Configuración")

tasa = st.sidebar.number_input(
    "Tasa de descuento (%)",
    min_value=0.0,
    value=10.0
)

modo = st.sidebar.radio(
    "Modo de ingreso",
    ["Ingreso Manual", "Subir Archivo"]
)

flujos = []

# =========================
# INGRESO MANUAL
# =========================
if modo == "Ingreso Manual":

    st.subheader("✍️ Ingresar Flujos Manualmente")

    cantidad = st.number_input(
        "Número de periodos",
        min_value=1,
        value=5
    )

    for i in range(cantidad):

        flujo = st.number_input(
            f"Flujo del periodo {i}",
            value=0.0,
            key=i
        )

        flujos.append(flujo)

# =========================
# SUBIR ARCHIVO
# =========================
else:

    archivo = st.file_uploader(
        "📂 Subir archivo",
        type=["csv", "xlsx", "txt"]
    )

    if archivo is not None:

        if archivo.name.endswith(".csv"):
            df = pd.read_csv(archivo, header=None)
            flujos = df.iloc[:, 0].tolist()

        elif archivo.name.endswith(".xlsx"):
            df = pd.read_excel(archivo)
            flujos = df.iloc[:, 0].tolist()

        elif archivo.name.endswith(".txt"):
            contenido = archivo.read().decode("utf-8")
            flujos = [float(x) for x in contenido.splitlines()]

# =========================
# MOSTRAR TABLA
# =========================
if flujos:

    st.subheader("📋 Flujos de Caja")

    tabla = pd.DataFrame({
        "Periodo": range(len(flujos)),
        "Flujo": flujos
    })

    st.dataframe(
        tabla,
        use_container_width=True
    )

    # =========================
    # BOTON CALCULAR
    # =========================
    if st.button("⚡ Calcular VAN"):

        tasa_decimal = tasa / 100

        van = 0

        for i, flujo in enumerate(flujos):
            van += flujo / ((1 + tasa_decimal) ** i)

        van = round(van, 2)

        rentable = van > 0

        # =========================
        # RESULTADO
        # =========================
        if rentable:
            mensaje = f"✅ VAN = {van} | PROYECTO RENTABLE"
        else:
            mensaje = f"❌ VAN = {van} | PROYECTO NO RENTABLE"

        st.markdown(
            f'<div class="result-box">{mensaje}</div>',
            unsafe_allow_html=True
        )

        # =========================
        # EXPORTAR EXCEL
        # =========================
        excel_temp = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".xlsx"
        )

        tabla["VAN"] = ""
        tabla.loc[0, "VAN"] = van

        tabla.to_excel(
            excel_temp.name,
            index=False
        )

        with open(excel_temp.name, "rb") as f:

            st.download_button(
                "📥 Descargar Excel",
                f,
                file_name="reporte_van.xlsx"
            )

        # =========================
        # EXPORTAR PDF
        # =========================
        pdf = FPDF()

        pdf.add_page()

        pdf.set_font("Arial", "B", 18)

        pdf.cell(
            200,
            10,
            "Reporte Financiero VAN",
            ln=True,
            align="C"
        )

        pdf.ln(10)

        pdf.set_font("Arial", "", 14)

        pdf.cell(
            200,
            10,
            f"VAN: {van}",
            ln=True
        )

        estado = (
            "Proyecto Rentable"
            if rentable
            else "Proyecto No Rentable"
        )

        pdf.cell(
            200,
            10,
            f"Resultado: {estado}",
            ln=True
        )

        pdf.ln(10)

        for i, flujo in enumerate(flujos):

            pdf.cell(
                200,
                10,
                f"Periodo {i}: {flujo}",
                ln=True
            )

        pdf_temp = tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".pdf"
        )

        pdf.output(pdf_temp.name)

        with open(pdf_temp.name, "rb") as f:

            st.download_button(
                "📄 Descargar PDF",
                f,
                file_name="reporte_van.pdf"
            )

Overwriting app.py


In [13]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-05-20 02:02:24.466 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.240.79:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://bright-laws-work.loca.lt
2026-05-20 02:09:28.926 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...
^C


In [14]:
!npm uninstall -g localtunnel -y
!pip install streamlit pyngrok --quiet
!wget -q -O - ipv4.icanhazip.com

⠙
up to date in 399ms
⠙34.139.240.79


In [17]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!npx cloudflared tunnel --url http://localhost:8501

⠙⠹⠸⠼⠴⠦Need to install the following packages:
cloudflared@0.7.1
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼2026-05-20T02:13:09Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-20T02:13:09Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-20T02:13:12Z INF +--------------------------------------------------------------------------------------------+
2026-05-20T02:13:12Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to